# Insectes : classification vs détection vs pose — où le modèle regarde-t-il ?Trois modèles YOLO26 entraînés sur les mêmes images avec trois supervisions différentes,ramenés à une même tâche : attribuer l'image à l'un des groupes.À lancer **après** `fuze_datasets.py` puis `create_background_class.py`.### Ce que la classe `background` change`detect` et `pose` peuvent **s'abstenir** : aucune boîte au-dessus du seuil. `cls` ne le peutpas — il est forcé de choisir un groupe même sur une image vide. La classe `background` luirend cette capacité, et le notebook traite **« cls prédit background » exactement comme« detect ne sort aucune boîte »**. Les trois modèles deviennent alors comparables sur le mêmeespace de décision : 3 groupes + abstention.### Deux méthodes d'explicabilité, volontairement redondantes- **Grad-CAM** — class-specific, rapide, mais passe par les gradients.- **Occlusion** — n'interroge le modèle que par son entrée/sortie. Plus lent, immunisé contre  les artefacts de gradient. **En cas de désaccord, c'est l'occlusion qui fait foi.**Puis le **test contre-factuel** sur les images sans insecte : la mesure la plus solide du lot.### Choix de simplification assumés| Choix | Conséquence ||---|---|| Letterbox commun aux 3 modèles | Le dataset `cls` est écrit letterboxé sur disque, donc le `Resize + CenterCrop` d'Ultralytics devient l'identité. Sans ça, `cls` verrait un recadrage central que `detect`/`pose` ne voient pas. || Masque = bbox seule | Surestime l'aire de l'insecte → l'EBPG paraît meilleur qu'il n'est. Biais **constant** entre les 3 modèles, donc sans effet sur leur comparaison. Ne pas publier l'EBPG en valeur absolue. || Grad-CAM seul (pas HiRes/Layer/Eigen) | On perd un garde-fou ; remplacé par le test de discriminativité de la cellule 6. || EBPG + pointing game seuls | Pas de mesure de *fidélité* (insertion/deletion). Une carte peut être bien localisée et décrire l'objet plutôt que la décision. |

## 1. ConfigurationSeule cellule à éditer.

In [ ]:
from pathlib import Pathimport numpy as np, pandas as pd, cv2, torch, yamlimport matplotlib.pyplot as pltfrom ultralytics import YOLOROOT = Path("./models/datasets")DATA = {    "cls":    ROOT / "AllSpecies-cls",                      # arborescence split/classe/    "detect": ROOT / "AllSpecies-detect" / "yolo-config.yaml",    "pose":   ROOT / "AllSpecies-pose"   / "yolo-config.yaml",}CF_DIR      = ROOT / "counterfactual"                       # cf. create_background_class.pyCF_VARIANTS = ["mean_noise", "telea", "gray"]               # ordre libreCF_TRAINED  = "mean_noise"    # variante VUE par cls a l'entrainement (cf. cellule 10)BACKGROUND  = "background"IMGSZ, EPOCHS, BATCH, SCALE, SEED = 640, 100, 16, "n", 0    # IMGSZ doit valoir celui des scriptsDEVICE = 0 if torch.cuda.is_available() else "cpu"SPLIT        = "test"N_PER_GROUP  = 10      # images expliquees par groupe (occlusion ~150 inferences/image)CONF         = 0.10    # seuil bas : on veut mesurer l'abstention, pas la masquerOCC_PATCH, OCC_STRIDE = 96, 48TARGET_LAYER = None    # None -> dernier bloc C2PSA du backbonetorch.manual_seed(SEED); np.random.seed(SEED)RUNS = Path("runs"); RUNS.mkdir(exist_ok=True)POSE_CFG = yaml.safe_load(open(DATA["pose"]))_names   = POSE_CFG["names"]GROUPS   = [_names[i] for i in sorted(_names)] if isinstance(_names, dict) else list(_names)NC, CHANCE = len(GROUPS), 1.0 / len(GROUPS)print("device:", DEVICE, "| groupes:", GROUPS)

## 2. Letterbox et masque insecteLe letterbox conserve le rapport d'aspect et complète en gris — **identique** à celui écrit surdisque par `fuze_datasets.py` pour le dataset `cls`, et à celui qu'Ultralytics applique eninterne pour `detect`/`pose`. C'est cette identité qui rend les cartes superposables.Les coordonnées YOLO normalisées se convertissent en pixels de l'image letterboxée enmultipliant par la taille redimensionnée puis en ajoutant le padding.

In [ ]:
def letterbox(img, size=IMGSZ):    """-> (image carree, ratio, pad_x, pad_y). Identique a fuze_datasets.letterbox_pil."""    h, w = img.shape[:2]    r = min(size / h, size / w)    nh, nw = max(1, int(round(h * r))), max(1, int(round(w * r)))    canvas = np.full((size, size, 3), 114, np.uint8)    px, py = (size - nw) // 2, (size - nh) // 2    canvas[py:py + nh, px:px + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)    return canvas, r, px, pydef load_square(path, size=IMGSZ):    img = cv2.imread(str(path))    if img is None:        raise FileNotFoundError(path)    if img.shape[0] == size and img.shape[1] == size:        return img          # deja letterboxee (dataset cls, images contre-factuelles)    return letterbox(img, size)[0]def bbox_mask(label_path, image_path, size=IMGSZ):    """Masque insecte dans le repere letterboxe."""    src = cv2.imread(str(image_path))    h, w = src.shape[:2]    r = min(size / h, size / w)    nh, nw = max(1, int(round(h * r))), max(1, int(round(w * r)))    px, py = (size - nw) // 2, (size - nh) // 2    m = np.zeros((size, size), np.uint8)    for line in Path(label_path).read_text().strip().splitlines():        p = line.split()        if len(p) < 5:            continue        cx, cy, bw, bh = map(float, p[1:5])        x1 = int((cx - bw / 2) * w * r) + px        y1 = int((cy - bh / 2) * h * r) + py        x2 = int((cx + bw / 2) * w * r) + px        y2 = int((cy + bh / 2) * h * r) + py        m[max(0, y1):max(0, y2), max(0, x1):max(0, x2)] = 255    return mdef norm01(a):    a = np.nan_to_num(a.astype(np.float32))    lo, hi = a.min(), a.max()    return np.zeros_like(a) if hi - lo < 1e-12 else (a - lo) / (hi - lo)def load_split(split=SPLIT):    """Liste (image, label, class_id, group) depuis le dataset pose."""    root  = Path(POSE_CFG["path"])    imdir = root / POSE_CFG.get(split, f"images/{split}")    rows = []    for img in sorted(imdir.rglob("*")):        if img.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".webp"}:            continue        lbl = Path(str(img).replace("/images/", "/labels/")).with_suffix(".txt")        if not lbl.exists() or not lbl.read_text().strip():            continue        cid = int(float(lbl.read_text().split()[0]))        rows.append({"image": str(img), "label": str(lbl),                     "class_id": cid, "group": GROUPS[cid]})    return pd.DataFrame(rows)test_df = load_split()print(f"{len(test_df)} images dans le split '{SPLIT}'")print(test_df.group.value_counts().to_string())

## 3. EntraînementHyperparamètres appariés : toute divergence autre que la supervision rendrait la comparaison ininterprétable.

In [ ]:
WEIGHTS = {"cls": f"yolo26{SCALE}-cls.pt",           "detect": f"yolo26{SCALE}.pt",           "pose": f"yolo26{SCALE}-pose.pt"}TASKS = ["cls", "detect", "pose"]def train(task):    out = RUNS / "train" / task / "weights" / "best.pt"    if out.exists():        print(f"{task}: deja entraine -> {out}")        return out    YOLO(WEIGHTS[task]).train(        data=str(DATA[task]), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,        device=DEVICE, project=str(RUNS / "train"), name=task, exist_ok=True,        deterministic=True,    )    return outmodels = {t: YOLO(str(train(t))) for t in TASKS}print({t: m.names for t, m in models.items()})

## 4. Prédiction image-level unifiée`detect` / `pose` : classe de la boîte la plus confiante ; aucune boîte = **abstention**.`cls` : classe la plus probable ; `background` = **abstention**.Le remappage passe par `model.names` et non par l'ordre des dossiers : Ultralytics ordonne lesclasses de classification **alphabétiquement**, ce qui ne coïncide pas forcément avec l'ordrede `GROUPS`. Avec des noms de groupes en minuscules, `background` passerait même en tête etdécalerait tous les indices. Ce remappage rend le notebook insensible à ce piège.

In [ ]:
_cls_names = models["cls"].names_cls_names = _cls_names if isinstance(_cls_names, dict) else dict(enumerate(_cls_names))NAME2IDX   = {n: i for i, n in _cls_names.items()}missing = [g for g in GROUPS if g not in NAME2IDX]assert not missing, f"Groupes absents du modele cls : {missing} (vu : {list(NAME2IDX)})"assert BACKGROUND in NAME2IDX, (    f"Classe '{BACKGROUND}' absente du modele cls. Lancer create_background_class.py "    "puis reentrainer, sinon cls ne peut pas s'abstenir.")CLS_COLS = [NAME2IDX[g] for g in GROUPS]      # colonnes des groupes dans la sortie clsCLS_BG   = NAME2IDX[BACKGROUND]print("cls :", _cls_names, "| colonnes groupes", CLS_COLS, "| background", CLS_BG)def model_class_index(task, group_idx):    """Indice de la classe dans l'espace de sortie du modele (Grad-CAM en a besoin)."""    return CLS_COLS[group_idx] if task == "cls" else group_idxdef predict(task, images, chunk=32):    """-> scores (N, NC) par groupe, abstain (N,) booleen."""    model, scores, abstain = models[task], [], []    for i in range(0, len(images), chunk):        kw = dict(imgsz=IMGSZ, verbose=False, device=DEVICE)        if task != "cls":            kw["conf"] = CONF        for r in model.predict(images[i:i + chunk], **kw):            if task == "cls":                p = r.probs.data.cpu().numpy().astype(np.float32)                scores.append(p[CLS_COLS])                abstain.append(bool(p.argmax() == CLS_BG))            else:                s = np.zeros(NC, np.float32)                empty = r.boxes is None or len(r.boxes) == 0                if not empty:                    c = r.boxes.cls.cpu().numpy().astype(int)                    f = r.boxes.conf.cpu().numpy()                    for k in range(NC):                        if (c == k).any():                            s[k] = f[c == k].max()                scores.append(s)                abstain.append(empty)    return np.stack(scores), np.array(abstain)def labels(scores, abstain):    return np.where(abstain, -1, scores.argmax(1))# --- performance sur les images reelles --------------------------------------imgs = [load_square(p) for p in test_df.image]y = test_df.class_id.to_numpy()rows = []for t in TASKS:    sc, ab = predict(t, imgs)    yp = labels(sc, ab)    decided = yp >= 0    rows.append({"task": t,                 "accuracy": (yp == y).mean(),                       # abstention = erreur                 "accuracy_si_decide": (yp[decided] == y[decided]).mean() if decided.any() else np.nan,                 "abstention": (~decided).mean(),                 "confiance_moy": sc.max(1).mean()})pd.DataFrame(rows).set_index("task").round(3)

## 5. Grad-CAMDeux subtilités que masquerait un appel naïf à `predict()` :1. `predict()` s'exécute sous `inference_mode` : aucun gradient. On appelle donc le réseau   directement.2. Pour récupérer le logit de classe sans dépendre de la valeur de retour de `forward` (qui   change selon la tâche et la version), on pose un hook sur `head.cv3`.**Point critique.** Sur une tête end-to-end (YOLO26), la branche `one2one_cv3` opère sur desfeatures **détachées** : aucun gradient ne peut la traverser jusqu'au backbone. La CAM estdonc nécessairement calculée sur la branche `one2many` (`cv3`), entraînée conjointement maispas identique à celle qui produit les prédictions. À mentionner en méthodologie.

In [ ]:
def get_net(task):    net = models[task].model.to(DEVICE if DEVICE != "cpu" else "cpu").float().eval()    for p in net.parameters():        p.requires_grad_(True)    return netdef get_layer(net):    if TARGET_LAYER is not None:        return net.model[TARGET_LAYER]    c2psa = [m for m in net.model if type(m).__name__.upper().startswith("C2PSA")]    assert c2psa, "Aucun bloc C2PSA : preciser TARGET_LAYER."    return c2psa[-1]      # dernier bloc du backbone = seul point commun aux 3 tachesdef gradcam(task, img, group_idx):    net = get_net(task)    head, layer = net.model[-1], get_layer(net)    cls_idx = model_class_index(task, group_idx)    store, cls_maps, handles = {}, [], []    def keep_act(m, i, o):          # ne rien renvoyer : un hook qui retourne une        store["a"] = o              # valeur remplacerait la sortie du module        if o.requires_grad:            o.register_hook(lambda g: store.__setitem__("g", g.detach()))    def keep_logits(m, i, o):        cls_maps.append(o)    handles.append(layer.register_forward_hook(keep_act))    if task != "cls":        handles += [b.register_forward_hook(keep_logits) for b in head.cv3]    x = torch.from_numpy(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).permute(2, 0, 1)[None]    x = (x.float() / 255).to(next(net.parameters()).device)    prev = head.training    head.training = True            # sorties brutes : evite softmax / postprocess NMS-free    try:        out = net(x)        if task == "cls":            o = out[0] if isinstance(out, (list, tuple)) else out            score = o.reshape(-1)[cls_idx]        else:            score = torch.cat([m[0, cls_idx].reshape(-1) for m in cls_maps]).max()        net.zero_grad(set_to_none=True)        score.backward()    finally:        head.training = prev        for h in handles:            h.remove()    a, g = store["a"][0].detach(), store.get("g")    assert g is not None, "Aucun gradient : verifier TARGET_LAYER."    cam = (g[0].mean((1, 2), keepdim=True) * a).sum(0).clamp(min=0).cpu().numpy()    return norm01(cv2.resize(cam, (IMGSZ, IMGSZ)))

## 6. Vérification : la CAM est-elle class-specific ?Le garde-fou minimal. Si les classes produisent la même carte, la couche cible est tropprofonde pour discriminer et les heatmaps ne signifient rien. Tant que cette cellule n'affichepas `OK` partout, ne pas interpréter la suite.

In [ ]:
probe = load_square(test_df.image.iloc[0])for t in TASKS:    maps = [gradcam(t, probe, c) for c in range(NC)]    ecart = np.mean([np.abs(maps[0] - m).mean() for m in maps[1:]]) if NC > 1 else 0.0    etat = "OK" if maps[0].std() > 1e-6 and ecart > 1e-4 else "PROBLEME"    print(f"{t:<7} ecart inter-classes {ecart:.5f}   [{etat}]")    if etat == "PROBLEME":        print(f"         -> essayer TARGET_LAYER parmi {list(get_net(t).model[-1].f)}")

## 7. OcclusionOn masque une fenêtre glissante et on mesure la chute du score de la vraie classe. Aucungradient, aucune hypothèse d'architecture : **strictement la même procédure pour les troistâches**. C'est la méthode de référence quand elle contredit Grad-CAM.

In [ ]:
def occlusion(task, img, group_idx, patch=OCC_PATCH, stride=OCC_STRIDE):    base = predict(task, [img])[0][0, group_idx]    coords, variants = [], []    for yy in range(0, IMGSZ - patch + 1, stride):        for xx in range(0, IMGSZ - patch + 1, stride):            v = img.copy()            v[yy:yy + patch, xx:xx + patch] = 114    # meme gris que le letterbox            variants.append(v); coords.append((yy, xx))    drops = base - predict(task, variants)[0][:, group_idx]    acc = np.zeros((IMGSZ, IMGSZ), np.float32)    cnt = np.zeros((IMGSZ, IMGSZ), np.float32)    for (yy, xx), d in zip(coords, drops):        acc[yy:yy + patch, xx:xx + patch] += d        cnt[yy:yy + patch, xx:xx + patch] += 1    return norm01(np.maximum(acc / np.maximum(cnt, 1), 0))print(f"{(((IMGSZ - OCC_PATCH)//OCC_STRIDE)+1)**2} inferences par image et par modele")

## 8. Métriques et boucle principale- **EBPG** — fraction de la masse de saillance dans l'insecte. `1 - EBPG` = dépendance au fond.- **pointing game** — l'argmax de la carte tombe-t-il dans l'insecte ?- **ratio** — EBPG normalisé par l'aire du masque. **C'est la colonne à lire** : un EBPG de  0.40 sur un insecte couvrant 40 % de l'image, c'est le hasard (ratio = 1).On explique toujours la **vraie** classe, pas la classe prédite : expliquer la classe préditemélangerait erreurs de classification et défauts de localisation.

In [ ]:
def ebpg(sal, mask):    tot = sal.sum()    return float(sal[mask > 0].sum() / tot) if tot > 0 else 0.0def pointing(sal, mask):    yy, xx = np.unravel_index(int(sal.argmax()), sal.shape)    return int(mask[yy, xx] > 0)sample = (test_df.groupby("group", group_keys=False)                 .apply(lambda d: d.sample(min(N_PER_GROUP, len(d)), random_state=SEED)))print(f"{len(sample)} images expliquees")rows, saliency = [], {}for t in TASKS:    for method, fn in (("gradcam", gradcam), ("occlusion", occlusion)):        for r in sample.itertuples():            img  = load_square(r.image)            mask = bbox_mask(r.label, r.image)            if mask.sum() == 0:                continue            sal = fn(t, img, r.class_id)            saliency[(t, method, r.image)] = sal            e, area = ebpg(sal, mask), (mask > 0).mean()            rows.append({"task": t, "method": method, "group": r.group,                         "ebpg": e, "pointing": pointing(sal, mask),                         "aire_masque": area, "ratio": e / area if area else np.nan})        print(f"{t}/{method} termine")sal_df = pd.DataFrame(rows)sal_df.groupby(["task", "method"])[["ebpg", "pointing", "ratio", "aire_masque"]].mean().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))for ax, (col, titre) in zip(axes, [("ebpg", "EBPG (masse de saillance sur l'insecte)"),                                   ("ratio", "Ratio EBPG / aire  (1 = hasard)")]):    sal_df.groupby(["task", "method"])[col].mean().unstack().plot(kind="bar", ax=ax, rot=0)    ref, lab = ((sal_df.aire_masque.mean(), "aire moyenne") if col == "ebpg" else (1, "hasard"))    ax.axhline(ref, ls="--", c="k", lw=1, label=lab)    ax.set_title(titre, fontsize=10); ax.legend(fontsize=8)plt.tight_layout(); plt.show()

## 9. Test contre-factuelLa mesure la plus solide du notebook : elle ne dépend d'aucune hypothèse d'explicabilité.Trois variantes produites par `create_background_class.py`, à lire **ensemble** :| variante | ce qu'elle fait | rôle ||---|---|---|| `mean_noise` | aplat + bruit + raccord flou | méthode d'entraînement de la classe `background` || `telea` | inpainting OpenCV, reconstruit le fond | **jamais vue à l'entraînement** || `gray` | aplat gris, aucun raccord | contrôle pur : réaction au trou |Si un modèle se comporte pareil sur `telea` et sur `gray`, il réagit à l'artefact, pas àl'absence d'insecte.

In [ ]:
def cf_images(variant, stems):    """Images contre-factuelles appariees par nom de fichier."""    d = CF_DIR / variant / SPLIT    out = {}    for p in d.glob("*"):        if p.suffix.lower() in {".png", ".jpg", ".jpeg"}:            out[p.stem] = p    return [out.get(s) for s in stems]stems = [Path(p).stem for p in test_df.image]avail = {v: cf_images(v, stems) for v in CF_VARIANTS}keep = [i for i in range(len(stems)) if all(avail[v][i] is not None for v in CF_VARIANTS)]print(f"{len(keep)}/{len(stems)} images appariees sur les {len(CF_VARIANTS)} variantes")assert keep, f"Aucune image trouvee dans {CF_DIR}. Lancer create_background_class.py."paired  = test_df.iloc[keep].reset_index(drop=True)y_true  = paired.class_id.to_numpy()ref_im  = [load_square(p) for p in paired.image]cf_rows = []for t in TASKS:    s_ref, a_ref = predict(t, ref_im); p_ref = labels(s_ref, a_ref)    for v in CF_VARIANTS:        s_cf, a_cf = predict(t, [load_square(avail[v][i]) for i in keep])        p_cf = labels(s_cf, a_cf)        cf_rows.append({            "task": t, "variante": v,            "dependance_au_fond": (p_cf == y_true).mean(),      # hasard = CHANCE            "abstention": a_cf.mean(),            "d_abstention": a_cf.mean() - a_ref.mean(),            "d_score_vraie_classe": (s_cf[np.arange(len(y_true)), y_true]                                     - s_ref[np.arange(len(y_true)), y_true]).mean(),            "flip_rate": (p_ref != p_cf).mean(),        })cf_df = pd.DataFrame(cf_rows)cf_df.round(3)

In [ ]:
piv = cf_df.pivot(index="task", columns="variante", values="dependance_au_fond")ax = piv.plot(kind="bar", rot=0, figsize=(7, 4))ax.axhline(CHANCE, ls="--", c="k", lw=1, label=f"hasard ({CHANCE:.2f})")ax.set_ylabel("classe encore correcte sans insecte"); ax.legend(fontsize=8)plt.tight_layout(); plt.show()print("Lecture :\n")for t in TASKS:    real, ctrl = float(piv.loc[t, "telea"]), float(piv.loc[t, "gray"])    if abs(real - ctrl) < 0.05:        v = "identique au controle gris -> reagit au trou, pas a l'absence d'insecte. NON CONCLUANT."    elif real <= CHANCE + 0.05:        v = "retombe au niveau du hasard -> pas de dependance au fond detectable."    elif real < 0.5:        v = "dependance au fond moderee mais superieure au hasard."    else:        v = "dependance au fond FORTE."    print(f"  {t:<7} telea {real:.3f} / gray {ctrl:.3f} -> {v}")

## 10. Diagnostic : `cls` a-t-il appris l'artefact ?Confond spécifique à la classe `background` : les images `mean_noise` sont dans**l'entraînement** de `cls`. Il peut donc avoir appris la signature de l'artefact (bruituniforme, raccord flou) au lieu d'avoir appris l'absence d'insecte.Le test tient en une comparaison : si `cls` s'abstient massivement sur `mean_noise` (vue) maispas sur `telea` (jamais vue), il reconnaît l'artefact, pas le vide. `detect` et `pose`, quin'ont jamais vu ces images, servent de témoins — un écart chez eux est du bruit, un écart chez`cls` seul est le symptôme.

In [ ]:
ab = cf_df.pivot(index="task", columns="variante", values="abstention")print(ab.round(3).to_string(), "\n")ecarts = {t: float(ab.loc[t, CF_TRAINED] - ab.loc[t, "telea"]) for t in TASKS}temoin = np.mean([ecarts[t] for t in TASKS if t != "cls"])print(f"ecart abstention ({CF_TRAINED} vue - telea jamais vue) :")for t in TASKS:    print(f"  {t:<7} {ecarts[t]:+.3f}")print(f"\nmoyenne des temoins (detect, pose) : {temoin:+.3f}")if ecarts["cls"] - temoin > 0.15:    print("\n=> cls s'abstient beaucoup plus sur la variante qu'il a vue a l'entrainement.\n"          "   Il a appris la SIGNATURE DE L'ARTEFACT, pas l'absence d'insecte.\n"          "   Ne pas interpreter son abstention comme une preuve qu'il regarde l'insecte.\n"          "   Correctif : entrainer la classe background sur un MELANGE de methodes\n"          "   (mean_noise + telea + gray), et garder une 4e methode inedite pour le test.")else:    print("\n=> pas de sur-abstention specifique a la variante d'entrainement :\n"          "   l'abstention de cls porte bien sur l'absence d'insecte.")

## 11. GalerieLigne = modèle, colonne = image. Grad-CAM et occlusion côte à côte : c'est leur **accord** qui est informatif, pas la beauté d'une carte isolée.

In [ ]:
def show(method, n=4):    picks = sample.sample(min(n, len(sample)), random_state=SEED)    fig, axes = plt.subplots(len(TASKS) + 1, len(picks),                             figsize=(3 * len(picks), 3 * (len(TASKS) + 1)), squeeze=False)    for j, r in enumerate(picks.itertuples()):        img, m = load_square(r.image), bbox_mask(r.label, r.image)        vis = img.copy()        x, yb, w, h = cv2.boundingRect((m > 0).astype(np.uint8))        cv2.rectangle(vis, (x, yb), (x + w, yb + h), (255, 255, 255), 2)        axes[0][j].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))        axes[0][j].set_title(r.group, fontsize=9)        if j == 0:            axes[0][j].set_ylabel("image", fontsize=10)        for i, t in enumerate(TASKS, start=1):            sal = saliency.get((t, method, r.image))            if sal is None:                continue            heat = cv2.applyColorMap((sal * 255).astype(np.uint8), cv2.COLORMAP_JET)            axes[i][j].imshow(cv2.cvtColor(cv2.addWeighted(heat, .45, img, .55, 0),                                           cv2.COLOR_BGR2RGB))            if j == 0:                axes[i][j].set_ylabel(t, fontsize=10)    for row in axes:        for a in row:            a.set_xticks([]); a.set_yticks([])    fig.suptitle(method, fontsize=12); plt.tight_layout(); plt.show()show("gradcam")show("occlusion")

## Comment lire les résultats1. **`ratio` avant `ebpg`.** Un EBPG élevé sur des insectes qui remplissent l'image ne veut   rien dire. `ratio > 1` = concentration sur l'insecte supérieure au hasard.2. **Grad-CAM vs occlusion.** S'ils convergent, la conclusion tient. S'ils divergent,   l'occlusion l'emporte : Grad-CAM passe par la branche `one2many` et par les gradients, deux   sources d'artefact que l'occlusion n'a pas.3. **Le contre-factuel prime sur les cartes.** Une heatmap dit où le modèle *semble* regarder ;   le contre-factuel dit s'il *a besoin* de l'insecte.4. **`d_abstention` se lit avec `dependance_au_fond`.** Un modèle qui reste correct sur les   images sans insecte *et* n'augmente pas son abstention s'appuie sur le fond. Un modèle qui   s'abstient massivement se comporte bien, même si son accuracy chute.5. **La cellule 10 conditionne la lecture de `cls`.** Si elle signale l'apprentissage de   l'artefact, l'abstention de `cls` ne prouve rien.### Limites- Grad-CAM des modèles `detect`/`pose` : branche `one2many` (la branche d'inférence `one2one`  reçoit des features détachées, aucun gradient possible).- Masque = bbox → EBPG optimiste en valeur absolue, mais comparable entre modèles.- La classe `background` déséquilibre le dataset `cls` si `BACKGROUND_RATIO` est trop haut ;  `report_balance()` dans `create_background_class.py` le signale.